## Hardened Structural Validator

In [ ]:
from typing import Dict, Any, List

import math 
REQUIRED_FIELDS = [
    "total_revenue",
    "fixed_cost_total",
    "variable_cost_total",
    "currency_code",
    "industry_type",
]
ALLOWED_CURRENCIES = ["USD", "INR", "JPY"]
def _is_valid_number(value) -> bool:
    if isinstance(value, bool):
        return False
    if not isinstance(value, (int, float)):
        return False
    if math.isnan(value) or math.isinf(value):
        return False
    return True

def hardened_structural_validation(payload: Dict[str, Any], rules: Dict[str, Any]) -> Dict[str, Any]:
    errors: List[Dict[str, Any]] = []
    warnings: List[Dict[str, Any]] = []

    # 1. Required fields check
    for field in REQUIRED_FIELDS:
        if field not in payload:
            errors.append(
                _build_error(
                    "MISSING_REQUIRED_FIELD",
                    f"{field} is required",
                    field,
                    SEVERITY_ERROR,
                )
            )
    if errors:
        return {
            "is_blocked": True,
            "errors": errors,
            "warnings": warnings,
        }
    revenue = payload["total_revenue"]
    fixed_cost = payload["fixed_cost_total"]
    variable_cost = payload["variable_cost_total"]
    currency = payload["currency_code"]
    industry = payload["industry_type"]        
    # 2. Numeric type enforcement
    numeric_fields = {
        "total_revenue": revenue,
        "fixed_cost_total": fixed_cost,
        "variable_cost_total": variable_cost,
    }
    for field, value in numeric_fields.items():
        if not _is_valid_number(value):
            errors.append(
                _build_error(
                    "INVALID_NUMERIC_TYPE",
                    f"{field} must be a finite int or float",
                    field,
                    SEVERITY_ERROR,
                )
            )
    # Stop if numeric types invalid
    if errors:
        return {
            "is_blocked": True,
            "errors": errors,
            "warnings": warnings,
        }
    # 3. Non-negative economic enforcement
    if revenue < 0:
        errors.append(
            _build_error(
                "NEGATIVE_REVENUE_NOT_ALLOWED",
                "Revenue cannot be negative",
                "total_revenue",
                SEVERITY_ERROR,
            )
        )            
    if fixed_cost < 0:
        errors.append(
            _build_error(
                "NEGATIVE_FIXED_COST_NOT_ALLOWED",
                "Fixed cost cannot be negative",
                "fixed_cost_total",
                SEVERITY_ERROR,
            )
        )

    if variable_cost < 0:
        errors.append(
            _build_error(
                "NEGATIVE_VARIABLE_COST_NOT_ALLOWED",
                "Variable cost cannot be negative",
                "variable_cost_total",
                SEVERITY_ERROR,
            )
        )
    # 5. Industry validation
    if not isinstance(industry, str) or industry not in INDUSTRY_CONFIG:
        errors.append(
            _build_error(
                "UNSUPPORTED_INDUSTRY",
                "Unsupported or invalid industry_type",
                "industry_type",
                SEVERITY_ERROR,
            )
        )

    # 6. Currency range enforcement (only if currency valid)
    # 4. Currency validation
    if not isinstance(currency, str) or currency not in ALLOWED_CURRENCIES:
       errors.append(
        _build_error(
            "UNSUPPORTED_CURRENCY",
            "Unsupported or invalid currency_code",
            "currency_code",
            SEVERITY_ERROR,
        )
    )
    # 6. Currency range enforcement
    if (
    isinstance(currency, str)
    and currency in rules["currency_ranges"]
    and _is_valid_number(revenue)
):
       low, soft_high, hard_high = rules["currency_ranges"][currency]

    if revenue < low or revenue > hard_high:
        errors.append(
            _build_error(
                "CURRENCY_OUT_OF_HARD_RANGE",
                "Revenue outside hard currency bounds",
                "total_revenue",
                SEVERITY_ERROR,
            )
        )
    elif revenue > soft_high:
        warnings.append(
            _build_error(
                "CURRENCY_SCALE_ANOMALY",
                "Revenue outside expected currency scale",
                "total_revenue",
                SEVERITY_WARNING,
            )
        )


    return {
        "is_blocked": any(e["severity"] == SEVERITY_ERROR for e in errors),
        "errors": errors,
        "warnings": warnings,
    }